# Model 3B: OPUS-MT Fine-Tuned Model

Colab-ready fine-tuning notebook for `Helsinki-NLP/opus-mt-en-ig`. It trains on every row in `Cleaned Data/Combined_train.jsonl`, evaluates on every row in `Cleaned Data/Combined_test.jsonl`, and saves the finished model plus CSV outputs back into `OPUS_MT Outputs`.


In [1]:
# Run this once in Google Colab if the packages are not already installed, then restart the runtime if prompted.
#%pip install -q -U transformers datasets accelerate sacrebleu sentencepiece sacremoses pandas==2.2.2

## 1. Imports and Configuration


In [2]:
from __future__ import annotations

import inspect
import json
import os
import random
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import DatasetDict, load_dataset
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

IN_COLAB = "google.colab" in sys.modules
MOUNT_DRIVE = True
SAVE_OUTPUTS_TO_DRIVE = True
COPY_DATA_TO_LOCAL = True
DRIVE_PROJECT_SUBDIR = "CS156 Final Assignment"
DRIVE_OUTPUT_SUBDIR = "OPUS_MT Outputs"

MODEL_NAME = "Helsinki-NLP/opus-mt-en-ig"
MODEL_LABEL = "fine_tuned"
RANDOM_SEED = 42

# None means every row from Combined_train.jsonl is used.
FINE_TUNE_TRAIN_SAMPLE_SIZE = None
FINE_TUNE_EPOCHS = 1

# Faster full-data defaults for Colab: large batches when possible, automatic downshift on OOM,
# mixed precision, shorter capped sequences, no generated validation metrics during training.
PER_DEVICE_TRAIN_BATCH_SIZE = 32
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1
AUTO_FIND_BATCH_SIZE = True
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.0
USE_ADAFACTOR = True
MAX_SOURCE_LENGTH = 96
MAX_TARGET_LENGTH = 96
TRAINING_EVAL_SAMPLE_SIZE = 2000
TOKENIZE_NUM_PROC = 2 if IN_COLAB else 1
DATALOADER_NUM_WORKERS = 2 if IN_COLAB else 0
GROUP_BY_LENGTH = True
LOGGING_STEPS = 100
SAVE_STEPS = 1000
EVAL_STEPS = 1000
SAVE_TOTAL_LIMIT = 2
RESUME_FROM_CHECKPOINT = True

# Final evaluation still runs generation on the complete Combined_test.jsonl file.
GENERATION_MAX_NEW_TOKENS = 80
GENERATION_NUM_BEAMS = 1
DIRECT_BATCH_SIZE = 32

EVAL_SET_SPECS = [
    {"label": "combined_test", "filename": "Combined_test.jsonl"},
]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = bool(torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8)
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)

print("Running in Colab:", IN_COLAB)
print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
print("FP16:", USE_FP16, "BF16:", USE_BF16)


Running in Colab: True
Python: /usr/bin/python3
PyTorch version: 2.10.0+cu128
Device: cuda
GPU: NVIDIA H100 80GB HBM3
CUDA: 12.8
FP16: False BF16: True


## 2. Project Paths


In [3]:
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB and MOUNT_DRIVE else None
DRIVE_PROJECT_DIR = DRIVE_ROOT / DRIVE_PROJECT_SUBDIR if DRIVE_ROOT is not None else None


def find_data_dir(project_dir):
    for data_dir_name in ("Cleaned Data", "Cleaned_Data"):
        data_dir = project_dir / data_dir_name
        if (data_dir / "Combined_train.jsonl").exists() and (data_dir / "Combined_test.jsonl").exists():
            return data_dir
    return None


PROJECT_DIR_CANDIDATES = []
if DRIVE_PROJECT_DIR is not None:
    PROJECT_DIR_CANDIDATES.append(DRIVE_PROJECT_DIR)

PROJECT_DIR_CANDIDATES.extend([
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    Path(r"C:\Users\Mr. Paul\Downloads\CS156 Final Assignment"),
])


def resolve_project_dir(candidates):
    for candidate in candidates:
        if find_data_dir(candidate) is not None:
            return candidate
    return None


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project directory. In Colab, put the project at "
        f"MyDrive/{DRIVE_PROJECT_SUBDIR} with Cleaned Data/Combined_train.jsonl and Combined_test.jsonl."
    )

SOURCE_DATA_DIR = find_data_dir(PROJECT_DIR)
if SOURCE_DATA_DIR is None:
    raise FileNotFoundError(f"Could not locate Cleaned Data under {PROJECT_DIR}.")

if IN_COLAB and COPY_DATA_TO_LOCAL:
    DATA_DIR = Path("/content/cs156_opus_data")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    required_filenames = ["Combined_train.jsonl", "Combined_test.jsonl", *[spec["filename"] for spec in EVAL_SET_SPECS]]
    for filename in sorted(set(required_filenames)):
        source_path = SOURCE_DATA_DIR / filename
        target_path = DATA_DIR / filename
        if not source_path.exists():
            raise FileNotFoundError(f"Missing data file: {source_path}")
        if not target_path.exists() or target_path.stat().st_size != source_path.stat().st_size:
            print(f"Copying {filename} to Colab local disk...")
            shutil.copy2(source_path, target_path)
else:
    DATA_DIR = SOURCE_DATA_DIR

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    if DRIVE_PROJECT_DIR is None:
        raise FileNotFoundError("Google Drive is not mounted, so outputs cannot be saved there.")
    OUTPUT_DIR = DRIVE_PROJECT_DIR / DRIVE_OUTPUT_SUBDIR
else:
    OUTPUT_DIR = PROJECT_DIR / DRIVE_OUTPUT_SUBDIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "2_fine_tuned_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "2_fine_tuned_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_INFO_PATH = OUTPUT_DIR / "2_fine_tuned_model_info.json"
REGISTRY_PATH = OUTPUT_DIR / "model_registry.json"

TRAIN_PATH = DATA_DIR / "Combined_train.jsonl"
TEST_PATH = DATA_DIR / "Combined_test.jsonl"
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Missing training file: {TRAIN_PATH}")
if not TEST_PATH.exists():
    raise FileNotFoundError(f"Missing test file: {TEST_PATH}")

eval_set_paths = [{**spec, "path": DATA_DIR / spec["filename"]} for spec in EVAL_SET_SPECS]
for spec in eval_set_paths:
    if not spec["path"].exists():
        raise FileNotFoundError(f"Missing evaluation file: {spec['path']}")

print("Project directory:", PROJECT_DIR)
print("Source data directory:", SOURCE_DATA_DIR)
print("Active data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)
print("Fine-tuned model directory:", MODEL_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Training path:", TRAIN_PATH)
print("Test path:", TEST_PATH)
for spec in eval_set_paths:
    print(f"Eval set: {spec['label']} -> {spec['path']}")


Mounted at /content/drive
Copying Combined_test.jsonl to Colab local disk...
Copying Combined_train.jsonl to Colab local disk...
Project directory: /content/drive/MyDrive/CS156 Final Assignment
Source data directory: /content/drive/MyDrive/CS156 Final Assignment/Cleaned Data
Active data directory: /content/cs156_opus_data
Output directory: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs
Fine-tuned model directory: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/2_fine_tuned_model
Checkpoint directory: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/2_fine_tuned_checkpoints
Training path: /content/cs156_opus_data/Combined_train.jsonl
Test path: /content/cs156_opus_data/Combined_test.jsonl
Eval set: combined_test -> /content/cs156_opus_data/Combined_test.jsonl


## 3. Load The Combined Train/Test Data


In [4]:
def load_jsonl_records(path, sample_size=None, seed=42):
    records = []
    rng = random.Random(seed)
    total_seen = 0

    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            item = json.loads(line)
            if "English" not in item or "Igbo" not in item:
                continue

            total_seen += 1
            if sample_size is None:
                records.append(item)
            elif len(records) < sample_size:
                records.append(item)
            else:
                replacement_index = rng.randint(0, total_seen - 1)
                if replacement_index < sample_size:
                    records[replacement_index] = item

    return records


def summarize_records(label, records):
    if not records:
        return {"split": label, "rows": 0, "avg_english_chars": 0.0, "avg_igbo_chars": 0.0}
    return {
        "split": label,
        "rows": len(records),
        "avg_english_chars": round(sum(len(row["English"]) for row in records) / len(records), 2),
        "avg_igbo_chars": round(sum(len(row["Igbo"]) for row in records) / len(records), 2),
    }


def summarize_hf_dataset(label, dataset, sample_size=2000):
    if len(dataset) == 0:
        return {"split": label, "rows": 0, "avg_english_chars": 0.0, "avg_igbo_chars": 0.0}

    sample_size = min(sample_size, len(dataset))
    sample = dataset.select(range(sample_size))
    return {
        "split": label,
        "rows": len(dataset),
        "avg_english_chars": round(sum(len(text) for text in sample["English"]) / sample_size, 2),
        "avg_igbo_chars": round(sum(len(text) for text in sample["Igbo"]) / sample_size, 2),
    }


raw_datasets = load_dataset(
    "json",
    data_files={"train": str(TRAIN_PATH), "test": str(TEST_PATH)},
)

train_raw_dataset = raw_datasets["train"]
test_raw_dataset = raw_datasets["test"]

if FINE_TUNE_TRAIN_SAMPLE_SIZE is not None:
    sample_size = min(FINE_TUNE_TRAIN_SAMPLE_SIZE, len(train_raw_dataset))
    train_raw_dataset = train_raw_dataset.shuffle(seed=RANDOM_SEED).select(range(sample_size))

monitor_eval_size = min(TRAINING_EVAL_SAMPLE_SIZE, len(test_raw_dataset))
monitor_eval_dataset = test_raw_dataset.shuffle(seed=RANDOM_SEED).select(range(monitor_eval_size))

eval_sets = []
for spec in eval_set_paths:
    records = load_jsonl_records(spec["path"])
    eval_sets.append({**spec, "records": records})

display(pd.DataFrame([
    summarize_hf_dataset("fine_tune_train_all_combined_train", train_raw_dataset),
    summarize_hf_dataset("training_monitor_subset_combined_test", monitor_eval_dataset),
    summarize_hf_dataset("final_eval_all_combined_test", test_raw_dataset),
    *[summarize_records(spec["label"], spec["records"]) for spec in eval_sets],
]))


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

,split,rows,avg_english_chars,avg_igbo_chars
0,fine_tune_train_all_combined_train,534064,88.75,93.31
1,training_monitor_subset_combined_test,2000,128.86,126.65
2,final_eval_all_combined_test,6390,129.61,127.08
3,combined_test,6390,127.84,125.86


## 4. Load Tokenizer and Base Model


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
# Avoid duplicate length-control warnings during generation.
model.generation_config.max_length = None
print("Loaded base model:", MODEL_NAME)
print("Tokenizer vocab size:", len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/819k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/800k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/294M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/294M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Loaded base model: Helsinki-NLP/opus-mt-en-ig
Tokenizer vocab size: 56127


## 5. Tokenize The Full Fine-Tuning Dataset


In [6]:
def preprocess_batch(batch):
    model_inputs = tokenizer(
        batch["English"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["Igbo"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


remove_train_columns = train_raw_dataset.column_names
remove_eval_columns = monitor_eval_dataset.column_names
map_num_proc = TOKENIZE_NUM_PROC if TOKENIZE_NUM_PROC and TOKENIZE_NUM_PROC > 1 else None

print("Tokenizing all training rows. This is cached by Hugging Face Datasets for this runtime.")
tokenized_train_dataset = train_raw_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=remove_train_columns,
    num_proc=map_num_proc,
    desc="Tokenizing train",
)

tokenized_monitor_eval_dataset = monitor_eval_dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=remove_eval_columns,
    num_proc=map_num_proc,
    desc="Tokenizing monitor eval",
)

print("Tokenized train rows:", len(tokenized_train_dataset))
print("Tokenized monitor eval rows:", len(tokenized_monitor_eval_dataset))


Tokenizing all training rows. This is cached by Hugging Face Datasets for this runtime.


Tokenizing train (num_proc=2):   0%|          | 0/534064 [00:00<?, ? examples/s]

Tokenizing monitor eval (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenized train rows: 534064
Tokenized monitor eval rows: 2000


## 6. Fine-Tune The Model


In [7]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)


def make_training_args(**kwargs):
    signature = inspect.signature(Seq2SeqTrainingArguments.__init__)
    valid_parameters = set(signature.parameters)
    if "eval_strategy" in valid_parameters and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    filtered_kwargs = {key: value for key, value in kwargs.items() if key in valid_parameters}
    ignored_kwargs = sorted(set(kwargs) - set(filtered_kwargs))
    if ignored_kwargs:
        print("Ignoring unsupported TrainingArguments for this Transformers version:", ignored_kwargs)
    return Seq2SeqTrainingArguments(**filtered_kwargs)


training_args = make_training_args(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=FINE_TUNE_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    logging_strategy="steps",
    logging_steps=LOGGING_STEPS,
    predict_with_generate=False,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="adafactor" if USE_ADAFACTOR else "adamw_torch",
    group_by_length=GROUP_BY_LENGTH,
    auto_find_batch_size=AUTO_FIND_BATCH_SIZE,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    report_to="none",
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_monitor_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

last_checkpoint = None
if RESUME_FROM_CHECKPOINT and CHECKPOINT_DIR.exists():
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
    if last_checkpoint is not None:
        print("Resuming from checkpoint:", last_checkpoint)

train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

model = trainer.model.to(device)
model.eval()
model.generation_config.max_length = None

model_info = {
    "model_label": MODEL_LABEL,
    "model_name": MODEL_NAME,
    "path": str(MODEL_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "description": "OPUS-MT model fine-tuned on every row in Combined_train.jsonl; final evaluation uses every row in Combined_test.jsonl.",
    "fine_tune_epochs": FINE_TUNE_EPOCHS,
    "train_sample_size": FINE_TUNE_TRAIN_SAMPLE_SIZE,
    "train_rows": len(train_raw_dataset),
    "monitor_eval_rows": len(monitor_eval_dataset),
    "max_source_length": MAX_SOURCE_LENGTH,
    "max_target_length": MAX_TARGET_LENGTH,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "generation_num_beams": GENERATION_NUM_BEAMS,
}
MODEL_INFO_PATH.write_text(json.dumps(model_info, indent=2), encoding="utf-8")

print("Saved fine-tuned model to:", MODEL_DIR)
print("Saved checkpoints to:", CHECKPOINT_DIR)
print("Saved model info to:", MODEL_INFO_PATH)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Ignoring unsupported TrainingArguments for this Transformers version: ['group_by_length']


Step,Training Loss,Validation Loss
1000,1.330363,3.488620
2000,1.307363,3.269523
3000,1.261629,3.119382
4000,1.208495,3.027050
5000,1.192489,2.938997
6000,1.170170,2.871125
7000,1.136422,2.816988
8000,1.111902,2.772774
9000,1.129728,2.719668
10000,1.136373,2.687602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/2_fine_tuned_model
Saved checkpoints to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/2_fine_tuned_checkpoints
Saved model info to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/2_fine_tuned_model_info.json


## 7. Update The Model Registry


In [8]:
registry = []
if REGISTRY_PATH.exists():
    registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))

registry = [row for row in registry if row.get("model_label") != MODEL_LABEL]
registry.append(model_info)
REGISTRY_PATH.write_text(json.dumps(registry, indent=2), encoding="utf-8")

display(pd.DataFrame(registry))
print("Updated model registry:", REGISTRY_PATH)


,model_label,model_name,path,checkpoint_dir,description,fine_tune_epochs,train_sample_size,train_rows,monitor_eval_rows,max_source_length,max_target_length,per_device_train_batch_size,gradient_accumulation_steps,generation_num_beams
0,fine_tuned,Helsinki-NLP/opus-mt-en-ig,/content/drive/MyDrive/CS156 Final Assignment/...,/content/drive/MyDrive/CS156 Final Assignment/...,OPUS-MT model fine-tuned on every row in Combi...,1,None,534064,2000,96,96,32,1,1


Updated model registry: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/model_registry.json


## 8. Translation and Evaluation Helpers


In [9]:
bleu_metric = BLEU()
chrf_metric = CHRF()


def translate_batch(texts, batch_size=DIRECT_BATCH_SIZE):
    outputs = []
    model.eval()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LENGTH,
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}

        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=GENERATION_MAX_NEW_TOKENS,
                num_beams=GENERATION_NUM_BEAMS,
            )

        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))

    return outputs


def evaluate_records(records, label):
    source_texts = [row["English"] for row in records]
    references = [row["Igbo"] for row in records]
    predictions = translate_batch(source_texts)

    refs_for_sacrebleu = [[text] for text in references]
    exact_match = sum(pred == ref for pred, ref in zip(predictions, references)) / len(predictions)
    bleu = bleu_metric.corpus_score(predictions, refs_for_sacrebleu).score
    chrf = chrf_metric.corpus_score(predictions, refs_for_sacrebleu).score

    metrics_df = pd.DataFrame(
        {
            "metric": ["Exact match", "BLEU", "chrF"],
            "value": [round(exact_match, 4), round(bleu, 2), round(chrf, 2)],
        }
    )
    preview_df = pd.DataFrame(
        {
            "English": source_texts,
            "Reference Igbo": references,
            "Predicted Igbo": predictions,
        }
    )

    metrics_path = OUTPUT_DIR / f"opus_mt_fine_tuned_{label}_metrics.csv"
    preview_path = OUTPUT_DIR / f"opus_mt_fine_tuned_{label}_preview.csv"
    metrics_df.to_csv(metrics_path, index=False)
    preview_df.to_csv(preview_path, index=False)
    return metrics_df, preview_df, metrics_path, preview_path


## 9. Evaluate The Fine-Tuned Model On The Full Combined Test Set


In [10]:
summaries = []

for spec in eval_sets:
    metrics_df, preview_df, metrics_path, preview_path = evaluate_records(spec["records"], spec["label"])
    summary = {"split": spec["label"], "model_variant": MODEL_LABEL, "rows": len(spec["records"])}
    summary.update({row["metric"]: row["value"] for _, row in metrics_df.iterrows()})
    summaries.append(summary)

    print("Saved metrics to:", metrics_path)
    print("Saved preview to:", preview_path)
    display(metrics_df)
    display(preview_df.head(10))

summary_df = pd.DataFrame(summaries)
summary_path = OUTPUT_DIR / "opus_mt_fine_tuned_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summary to:", summary_path)
display(summary_df)


Saved metrics to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/opus_mt_fine_tuned_combined_test_metrics.csv
Saved preview to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/opus_mt_fine_tuned_combined_test_preview.csv


,metric,value
0,Exact match,0.0074
1,BLEU,39.6700
2,chrF,56.5600


,English,Reference Igbo,Predicted Igbo
0,The LORD God made a woman from the rib which h...,"Mgbe ahụ, Onyenwe anyị Chineke ji otu ọgịrịga ...",Onyenwe anyị Chineke mere nwanyị n'akụkụ ọgịrị...
1,Others jailed includes; Adamu Mohammed who is ...,Ụfọdụ ndị ọzọ atụrụ mkpọrọ bụ Adamu Mohammed d...,Ndị ọzọ a tụrụ mkpọrọ gụnyere Adamu Mohammed o...
2,For who makes you different? And what do you h...,Onye gwara gị na ị pụrụ iche n'etiti mmadụ ndị...,Ma onye mere ka unu dị iche? Gịnị ka unu nwere...
3,"you want to buy something, go to these places","Ị chọrọ izu ihe, gaa ebe ndị a","Ị chọrọ ịzụta ihe, gaa n'ebe ndị a"
4,This company has stopped work there and sacked...,Ụlọọrụ a akwụsịla ọrụ n'ebe ahụ ma chụpụ ndị ọ...,Ụlọ ọrụ a akwụsịla ọrụ ebe ahụ ma chụrụ ndị ọr...
5,"After about one hour passed, another confident...","Mgbe ihe dị ka otu awa gafere, onye ọzọ bịakwa...","Mgbe ihe dị ka otu awa gasịrị, onye ọzọ ji obi..."
6,"He found a fresh jawbone of a donkey, put out ...","Mgbe ahụ Samsin lere anya gburugburu, hụ ọkpụk...","Ọ hụrụ otu eriri ịnyịnya ibu ọhụrụ, jide ya, w..."
7,But he abandoned the counsel of the old men wh...,"Ma ọ nabataghị ndụmọdụ ndị okenye nyere ya, ọ ...",Ma ọ hapụrụ ndụmọdụ nke ndị okenye ha nyere ya...
8,The Federal Government could have simply incre...,Gọọmentị etiti gaara enyekwu ikike nye ụlọ ọrụ...,Gọọmenti etiti gaara eme ka ndị ọrụ ngo nweta ...
9,"Stop Attacking Shoprite In Nigeria, They're Ow...",Kwụsị ịwakpo oke ụlọahịa na mba Naịjirịa. Ọ bụ...,"Kwụsị ịwakpo ndị ahịa na Naịjirịa, ha bụ ndị N..."


Saved summary to: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs/opus_mt_fine_tuned_summary.csv


,split,model_variant,rows,Exact match,BLEU,chrF
0,combined_test,fine_tuned,6390,0.0074,39.67,56.56


## 10. Manual Translation Check


In [11]:
def translate_text(text):
    return translate_batch([text], batch_size=1)[0]


english_text = "How are you?"
predicted_igbo = translate_text(english_text)

print("Model variant:", MODEL_LABEL)
print("English:", english_text)
print("Predicted Igbo:", predicted_igbo)


Model variant: fine_tuned
English: How are you?
Predicted Igbo: Olee otú ị bụ?


## 11. Saved Artifact List


In [12]:
saved_files = [
    {"artifact": "fine-tuned model directory", "path": str(MODEL_DIR)},
    {"artifact": "checkpoint directory", "path": str(CHECKPOINT_DIR)},
    {"artifact": "model info", "path": str(MODEL_INFO_PATH)},
    {"artifact": "model registry", "path": str(REGISTRY_PATH)},
    {"artifact": "evaluation summary", "path": str(OUTPUT_DIR / "opus_mt_fine_tuned_summary.csv")},
]
for spec in EVAL_SET_SPECS:
    label = spec["label"]
    saved_files.append({"artifact": f"{label} metrics", "path": str(OUTPUT_DIR / f"opus_mt_fine_tuned_{label}_metrics.csv")})
    saved_files.append({"artifact": f"{label} preview", "path": str(OUTPUT_DIR / f"opus_mt_fine_tuned_{label}_preview.csv")})

display(pd.DataFrame(saved_files))
print("If this ran in Colab with Drive mounted, these files are already back in Google Drive under:", OUTPUT_DIR)


,artifact,path
0,fine-tuned model directory,/content/drive/MyDrive/CS156 Final Assignment/...
1,checkpoint directory,/content/drive/MyDrive/CS156 Final Assignment/...
2,model info,/content/drive/MyDrive/CS156 Final Assignment/...
3,model registry,/content/drive/MyDrive/CS156 Final Assignment/...
4,evaluation summary,/content/drive/MyDrive/CS156 Final Assignment/...
5,combined_test metrics,/content/drive/MyDrive/CS156 Final Assignment/...
6,combined_test preview,/content/drive/MyDrive/CS156 Final Assignment/...


If this ran in Colab with Drive mounted, these files are already back in Google Drive under: /content/drive/MyDrive/CS156 Final Assignment/OPUS_MT Outputs
